In [ ]:
# ============================================================
# STOCHASTIC SQP-PINN (JAX) FOR NUCLEAR THERMAL COUPLING
# One NN outputs: [phi, Ts, u, v, p, Tf]
#
# NOTES:
#   - phi BC/IC match neutron MOOSE input exactly
#   - right fluid boundary uses MOOSE VALUE targets:
#         u=u_M, v=v_M, p=p_M, Tf=Tf_M
#   - old fluid case:
#         u(x,y,0)=1, v(x,y,0)=1e-12, Tf(x,y,0)=560
#         inlet: u=0, v=0.4, Tf=560
# ============================================================

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_command_buffer="

import time
import math
from functools import partial
import numpy as np

import jax
import jax.numpy as jnp
from jax import random, vmap, jacrev, hessian

# ============================================================
# Precision / dtype
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ============================================================
# USER SETTINGS / BENCHMARK
# ============================================================
Ls = DTYPE(0.0076)
Lf = DTYPE(0.0114)
Ly = DTYPE(0.75)
T_end = DTYPE(5.0)

x_min, x_max = DTYPE(0.0), DTYPE(Ls + Lf)
y_min, y_max = DTYPE(0.0), DTYPE(Ly)
t_min, t_max = DTYPE(0.0), DTYPE(T_end)

rho_f = DTYPE(11096.0)
rho_s = DTYPE(16020.0)

V_INLET_BC = DTYPE(0.4)
T_INLET = DTYPE(560.0)
P_OUTLET = DTYPE(0.0)
POWER_COEF = DTYPE(5e7)

# old fluid case
FLUID_U_IC = DTYPE(1.0)
FLUID_V_IC = DTYPE(1e-12)
FLUID_T_IC = DTYPE(560.0)

v_neu   = DTYPE(2.416)
D_fuel  = DTYPE(0.008249)
D_fluid = DTYPE(0.01)

# phi from neutron MOOSE file
PHI_TOPBOT_VAL = DTYPE(0.5)
PHI_RIGHT_VAL  = DTYPE(0.0)

MU_DAMP_FIXED = DTYPE(1e-4)

PHI_TXT_PATH = "./phi.txt"
TFLUID_PATH  = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_Tfluid.npy"

X_MARGIN = DTYPE(2e-5)

# ============================================================
# OUTPUT SCALES
# ============================================================
# IMPORTANT: phi can be much larger than 7 in the solution field
PHI_OUT_SCALE = DTYPE(15.0)
TS_OUT_SCALE  = DTYPE(350.0)
TF_OUT_SCALE  = DTYPE(250.0)
U_OUT_SCALE   = DTYPE(1.0)
V_OUT_SCALE   = DTYPE(0.5)
P_OUT_SCALE   = DTYPE(7.0)

PHI_VAL_SCALE = PHI_OUT_SCALE
TS_VAL_SCALE  = TS_OUT_SCALE
TF_VAL_SCALE  = TF_OUT_SCALE
U_VAL_SCALE   = U_OUT_SCALE
V_VAL_SCALE   = V_OUT_SCALE
P_VAL_SCALE   = P_OUT_SCALE
T_IF_VAL_SCALE = jnp.maximum(TS_VAL_SCALE, TF_VAL_SCALE)

# ============================================================
# SAMPLE COUNTS
# ============================================================
NX_OBJ_PHI, NY_OBJ_PHI, NT_OBJ_PHI = 20, 20, 10
NX_OBJ_S,   NY_OBJ_S,   NT_OBJ_S   = 20, 20, 10
NX_OBJ_F,   NY_OBJ_F,   NT_OBJ_F   = 20, 20, 10

NX_CON_PHI, NY_CON_PHI, NT_CON_PHI = 4, 4, 4
NX_CON_S,   NY_CON_S,   NT_CON_S   = 4, 4, 4
NX_CON_F,   NY_CON_F,   NT_CON_F   = 5, 5, 5

K_CON_PHI = NX_CON_PHI * NY_CON_PHI * NT_CON_PHI
K_CON_S   = NX_CON_S   * NY_CON_S   * NT_CON_S
K_CON_F   = NX_CON_F   * NY_CON_F   * NT_CON_F

N_PER_CELL = 1
N_BC_PER_BIN = 1
N_IC_PER_BIN = 1
N_IF_PER_BIN = 1

# separate BC counts
K_BC_PHI_LEFT   = 50
K_BC_PHI_RIGHT  = 30
K_BC_PHI_Y0     = 30
K_BC_PHI_Y1     = 30

K_BC_TS_X0      = 40

K_BC_F_INLET    = 60
K_BC_F_OUTLET   = 60
K_BC_F_RIGHT    = 0

NX_IC, NY_IC = 15, 20
K_IC = NX_IC * NY_IC

NY_IF, NT_IF = 10, 10
K_IF = NY_IF * NT_IF

# ============================================================
# BLOCK SIZES
# ============================================================
N_PDE_PHI = K_CON_PHI
N_PDE_TS  = K_CON_S
N_PDE_FLD = 4 * K_CON_F  # cont, ru, rv, rTf

N_BC_PHI = K_BC_PHI_LEFT + K_BC_PHI_RIGHT + K_BC_PHI_Y0 + K_BC_PHI_Y1
N_BC_SOL = K_BC_TS_X0
N_BC_FLD = 3 * K_BC_F_INLET + K_BC_F_OUTLET + 4 * K_BC_F_RIGHT
N_IF     = 2 * K_IF
N_IC     = 5 * K_IC

M_CON = N_PDE_PHI + N_PDE_TS + N_PDE_FLD + N_BC_PHI + N_BC_SOL + N_BC_FLD + N_IF + N_IC

# ============================================================
# LOAD phi.txt
# ============================================================
def load_phi_piecewise_multilinear(path: str):
    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    iy = lines.index("AXIS Y")
    it = lines.index("AXIS T")
    idata = lines.index("DATA")

    y_tokens = []
    k = iy + 1
    while k < len(lines) and lines[k] != "AXIS T":
        for tok in lines[k].split():
            try:
                float(tok)
                y_tokens.append(tok)
            except Exception:
                pass
        k += 1
    y = np.array([float(s) for s in y_tokens], dtype=np.float64)

    t_tokens = []
    k = it + 1
    while k < len(lines) and lines[k] != "DATA":
        for tok in lines[k].split():
            try:
                float(tok)
                t_tokens.append(tok)
            except Exception:
                pass
        k += 1
    t = np.array([float(s) for s in t_tokens], dtype=np.float64)

    Ny = len(y)
    Nt = len(t)

    data_tokens = []
    for ln in lines[idata + 1:]:
        for tok in ln.split():
            data_tokens.append(float(tok))
    data = np.array(data_tokens, dtype=np.float64)

    if data.size != Ny * Nt:
        raise ValueError(f"phi.txt DATA size mismatch: got {data.size}, expected {Ny * Nt}")

    Z_t_y = data.reshape(Nt, Ny)
    Z_y_t = Z_t_y.T
    return y, t, Z_y_t

y_np, t_np, Z_np = load_phi_piecewise_multilinear(PHI_TXT_PATH)
PHI_Y = jnp.array(y_np, dtype=DTYPE)
PHI_T = jnp.array(t_np, dtype=DTYPE)
PHI_Z = jnp.array(Z_np, dtype=DTYPE)

print("Loaded phi.txt:", y_np.shape, t_np.shape, Z_np.shape, "range=", float(Z_np.min()), float(Z_np.max()))

@jax.jit
def interp_yt(y, t, y_grid, t_grid, Z):
    y = jnp.asarray(y, dtype=DTYPE)
    t = jnp.asarray(t, dtype=DTYPE)

    Ny = y_grid.shape[0]
    Nt = t_grid.shape[0]

    iy = jnp.clip(jnp.searchsorted(y_grid, y, side="right") - 1, 0, Ny - 2)
    it = jnp.clip(jnp.searchsorted(t_grid, t, side="right") - 1, 0, Nt - 2)

    y0 = y_grid[iy]
    y1 = y_grid[iy + 1]
    t0 = t_grid[it]
    t1 = t_grid[it + 1]

    wy = (y - y0) / (y1 - y0 + EPS)
    wt = (t - t0) / (t1 - t0 + EPS)

    z00 = Z[iy, it]
    z10 = Z[iy + 1, it]
    z01 = Z[iy, it + 1]
    z11 = Z[iy + 1, it + 1]

    z0 = z00 + wy * (z10 - z00)
    z1 = z01 + wy * (z11 - z01)
    return z0 + wt * (z1 - z0)

def phi_left_target(y, t):
    t_clip = jnp.clip(t, PHI_T[0], PHI_T[-1])
    return interp_yt(y, t_clip, PHI_Y, PHI_T, PHI_Z)

# ============================================================
# LOAD MOOSE FLUID TRUTH
# ============================================================
SYNC_TIMES = np.array([
    0.0, 0.3125, 0.625, 0.9375, 1.25, 1.5625, 1.875, 2.1875,
    2.5, 2.8125, 3.125, 3.4375, 3.75, 4.0625, 4.375, 4.6875, 5.0
], dtype=np.float64)

def load_tfluid_truth(path: str, case_idx: int = 0):
    arr = np.load(path)

    # saved as (Ncase, 4, Nt, Nx_f, Ny)
    if arr.ndim == 5:
        arr = arr[case_idx]

    if arr.ndim != 4 or arr.shape[0] != 4:
        raise ValueError(f"Expected shape (4, Nt, Nx_f, Ny), got {arr.shape}")

    return arr

tfluid_np = load_tfluid_truth(TFLUID_PATH)
print("Loaded tfluid truth:", tfluid_np.shape)

# read order from exporter: [T_fluid, pressure, vel_x, vel_y]
Tf_true_np = tfluid_np[0]   # (Nt, Nx_f, Ny)
p_true_np  = tfluid_np[1]
u_true_np  = tfluid_np[2]
v_true_np  = tfluid_np[3]

Nt_true, Nx_f_true, Ny_true = u_true_np.shape
if Nt_true != len(SYNC_TIMES):
    raise ValueError(f"SYNC_TIMES length {len(SYNC_TIMES)} != Nt_true {Nt_true}")

Y_FLUID_np = np.linspace(0.0, float(Ly), Ny_true, dtype=np.float64)

# right boundary values: shape (Nt, Ny)
U_RIGHT_VAL_np  = u_true_np[:, -1, :]
V_RIGHT_VAL_np  = v_true_np[:, -1, :]
P_RIGHT_VAL_np  = p_true_np[:, -1, :]
TF_RIGHT_VAL_np = Tf_true_np[:, -1, :]

Y_FLUID = jnp.array(Y_FLUID_np, dtype=DTYPE)
T_SYNC  = jnp.array(SYNC_TIMES, dtype=DTYPE)

# store as (Ny, Nt) for interpolation
U_RIGHT_VAL  = jnp.array(U_RIGHT_VAL_np.T,  dtype=DTYPE)
V_RIGHT_VAL  = jnp.array(V_RIGHT_VAL_np.T,  dtype=DTYPE)
P_RIGHT_VAL  = jnp.array(P_RIGHT_VAL_np.T,  dtype=DTYPE)
TF_RIGHT_VAL = jnp.array(TF_RIGHT_VAL_np.T, dtype=DTYPE)

def u_right_target(y, t):
    return interp_yt(y, jnp.clip(t, T_SYNC[0], T_SYNC[-1]), Y_FLUID, T_SYNC, U_RIGHT_VAL)

def v_right_target(y, t):
    return interp_yt(y, jnp.clip(t, T_SYNC[0], T_SYNC[-1]), Y_FLUID, T_SYNC, V_RIGHT_VAL)

def p_right_target(y, t):
    return interp_yt(y, jnp.clip(t, T_SYNC[0], T_SYNC[-1]), Y_FLUID, T_SYNC, P_RIGHT_VAL)

def Tf_right_target(y, t):
    return interp_yt(y, jnp.clip(t, T_SYNC[0], T_SYNC[-1]), Y_FLUID, T_SYNC, TF_RIGHT_VAL)

# ============================================================
# MATERIAL LAWS
# ============================================================
def safe_T(T):
    return jnp.clip(T, DTYPE(300.0), DTYPE(1200.0))

def mu_f_fun(T):
    T = safe_T(T)
    return DTYPE(4.94e-4) * jnp.exp(DTYPE(754.1) / (T + EPS))

def k_f_fun(T):
    T = safe_T(T)
    return DTYPE(3.61) + DTYPE(1.517e-2) * T - DTYPE(1.741e-6) * T * T

def cp_f_fun(T):
    T = safe_T(T)
    return DTYPE(159.0) - DTYPE(2.72e-2) * T + DTYPE(7.12e-6) * T * T

def cp_s_fun(T):
    T = safe_T(T)
    return (DTYPE(1.359) + DTYPE(0.05812) * T + DTYPE(1.086e6) / (T * T + EPS)) * DTYPE(5.0)

def k_s_fun(T):
    T = safe_T(T)
    c0 = DTYPE(17.5) * (DTYPE(1.0) - DTYPE(0.223)) / (DTYPE(1.0) + DTYPE(0.161))
    c1 = DTYPE(1.54e-2) * (DTYPE(1.0) + DTYPE(0.0061)) / (DTYPE(1.0) + DTYPE(0.161))
    c2 = DTYPE(9.38e-6)
    return c0 + c1 * T + c2 * T * T

def sigma_af_fuel(T):
    T = safe_T(T)
    return (
        DTYPE(2.416) * DTYPE(583.5) * DTYPE(1.305) * DTYPE(1.602) * DTYPE(0.1)
        - (DTYPE(13.47) * (T - DTYPE(560.0)) / (DTYPE(900.0) - DTYPE(560.0)) + DTYPE(7.53))
        * DTYPE(2.1479) * DTYPE(1.602)
        + DTYPE(0.185) * DTYPE(6.6072) * DTYPE(0.1)
        - DTYPE(680.9) * DTYPE(1.305) * DTYPE(1.602) * DTYPE(0.1)
    )

def sigma_af_fluid(T):
    T = safe_T(T)
    return -(DTYPE(20.0) + DTYPE(20.0) * (T - DTYPE(560.0)) / (DTYPE(800.0) - DTYPE(560.0)))

def dk_s_dT(T):
    T = safe_T(T)
    c1 = DTYPE(1.54e-2) * (DTYPE(1.0) + DTYPE(0.0061)) / (DTYPE(1.0) + DTYPE(0.161))
    c2 = DTYPE(9.38e-6)
    return c1 + DTYPE(2.0) * c2 * T

def dk_f_dT(T):
    T = safe_T(T)
    a = DTYPE(1.517e-2)
    b = DTYPE(1.741e-6)
    return a - DTYPE(2.0) * b * T

def dmu_f_dT(T):
    T = safe_T(T)
    mu = mu_f_fun(T)
    return mu * (-DTYPE(754.1)) / (T * T + EPS)

def div_k_grad_scalar(k, dk_dT_fun, T, Tx, Ty, Txx, Tyy):
    return k * (Txx + Tyy) + dk_dT_fun(T) * (Tx * Tx + Ty * Ty)

def div_mu_grad_component(mu, dmu_dT_fun, Tf, Tfx, Tfy, ux, uy, uxx, uyy):
    return mu * (uxx + uyy) + dmu_dT_fun(Tf) * (Tfx * ux + Tfy * uy)

# ============================================================
# SCALES
# ============================================================
Lx       = (x_max - x_min) + EPS
Ly_len   = (y_max - y_min) + EPS
Lx_fluid = (x_max - Ls) + EPS
Lx_solid = (Ls - x_min) + EPS
Lmin     = jnp.minimum(Lx, Ly_len)

GRAD_TSX_SCALE = TS_VAL_SCALE / (Lx_solid + EPS)

D_MAX    = jnp.maximum(D_fuel, D_fluid)
SIG_REF  = jnp.maximum(jnp.abs(sigma_af_fuel(T_INLET)), jnp.abs(sigma_af_fluid(T_INLET)))
K_S_REF  = k_s_fun(T_INLET)
K_F_REF  = k_f_fun(T_INLET)
CP_S_REF = cp_s_fun(T_INLET)
CP_F_REF = cp_f_fun(T_INLET)
MU_REF   = mu_f_fun(T_INLET)

FLUX_SCALE = jnp.maximum(
    K_S_REF * TS_VAL_SCALE / (Lx_solid + EPS),
    K_F_REF * TF_VAL_SCALE / (Lx_fluid + EPS),
) + EPS

PHI_TIME_REF  = PHI_VAL_SCALE / (v_neu * (T_end + EPS))
PHI_DIFF_REF  = D_MAX * PHI_VAL_SCALE / (Lmin**2 + EPS)
PHI_REAC_REF  = SIG_REF * PHI_VAL_SCALE
PHI_RES_SCALE = PHI_TIME_REF + PHI_DIFF_REF + PHI_REAC_REF + EPS

TS_TIME_REF  = rho_s * CP_S_REF * TS_VAL_SCALE / (T_end + EPS)
TS_DIFF_REF  = K_S_REF * TS_VAL_SCALE / (Lx_solid**2 + EPS)
TS_SRC_REF   = POWER_COEF * PHI_VAL_SCALE
TS_RES_SCALE = TS_TIME_REF + TS_DIFF_REF + TS_SRC_REF + EPS

CONT_SCALE = jnp.maximum(
    U_VAL_SCALE / (Lx_fluid + EPS),
    V_VAL_SCALE / (Ly_len + EPS),
) + EPS

MOMX_TIME_REF  = rho_f * U_VAL_SCALE / (T_end + EPS)
MOMX_ADV_REF   = rho_f * (
    U_VAL_SCALE * U_VAL_SCALE / (Lx_fluid + EPS)
    + V_VAL_SCALE * U_VAL_SCALE / (Ly_len + EPS)
)
MOMX_P_REF     = P_VAL_SCALE / (Lx_fluid + EPS)
MOMX_VIS_REF   = MU_REF * (
    U_VAL_SCALE / (Lx_fluid**2 + EPS)
    + U_VAL_SCALE / (Ly_len**2 + EPS)
)
MOMX_RES_SCALE = MOMX_TIME_REF + MOMX_ADV_REF + MOMX_P_REF + MOMX_VIS_REF + EPS

MOMY_TIME_REF  = rho_f * V_VAL_SCALE / (T_end + EPS)
MOMY_ADV_REF   = rho_f * (
    U_VAL_SCALE * V_VAL_SCALE / (Lx_fluid + EPS)
    + V_VAL_SCALE * V_VAL_SCALE / (Ly_len + EPS)
)
MOMY_P_REF     = P_VAL_SCALE / (Ly_len + EPS)
MOMY_VIS_REF   = MU_REF * (
    V_VAL_SCALE / (Lx_fluid**2 + EPS)
    + V_VAL_SCALE / (Ly_len**2 + EPS)
)
MOMY_RES_SCALE = MOMY_TIME_REF + MOMY_ADV_REF + MOMY_P_REF + MOMY_VIS_REF + EPS

TF_TIME_REF  = rho_f * CP_F_REF * TF_VAL_SCALE / (T_end + EPS)
TF_ADV_REF   = rho_f * CP_F_REF * (
    U_VAL_SCALE * TF_VAL_SCALE / (Lx_fluid + EPS)
    + V_VAL_SCALE * TF_VAL_SCALE / (Ly_len + EPS)
)
TF_DIFF_REF  = K_F_REF * TF_VAL_SCALE * (
    1.0 / (Lx_fluid**2 + EPS)
    + 1.0 / (Ly_len**2 + EPS)
)
TF_RES_SCALE = TF_TIME_REF + TF_ADV_REF + TF_DIFF_REF + EPS

# ============================================================
# TARGETS / ICS
# ============================================================
@jax.jit
def phi_ic(y):
    # correct from MOOSE
    return DTYPE(2.0) * jnp.cos(DTYPE(3.14) * (y - DTYPE(0.375)) / DTYPE(0.75))

def inlet_v_profile(x):
    return DTYPE(0.4) * jnp.ones_like(x)

# ============================================================
# NETWORK
# ============================================================
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params

def normalize_xyt(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    t = X[:, 2:3]

    x_n = 2.0 * (x - x_min) / (x_max - x_min + EPS) - 1.0
    y_n = 2.0 * (y - y_min) / (y_max - y_min + EPS) - 1.0
    t_n = 2.0 * (t - t_min) / (t_max - t_min + EPS) - 1.0
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)

    phi_hat = h[:, 0:1]
    Ts_hat  = h[:, 1:2]
    u_hat   = h[:, 2:3]
    v_hat   = h[:, 3:4]
    p_hat   = h[:, 4:5]
    Tf_hat  = h[:, 5:6]

    phi = PHI_OUT_SCALE * phi_hat
    Ts  = T_INLET + TS_OUT_SCALE * Ts_hat
    u   = U_OUT_SCALE * u_hat
    v   = V_OUT_SCALE * v_hat
    p   = P_OUT_SCALE * p_hat
    Tf  = T_INLET + TF_OUT_SCALE * Tf_hat

    return jnp.concatenate([phi, Ts, u, v, p, Tf], axis=1)

def flatten_params(params):
    flat_parts = []
    shapes = []
    for layer in params:
        W, b = layer["W"], layer["b"]
        flat_parts.append(W.reshape(-1))
        flat_parts.append(b.reshape(-1))
        shapes.append((W.shape, b.shape))
    theta = jnp.concatenate(flat_parts).astype(DTYPE)
    return theta, tuple(shapes)

def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)
        W = theta[idx: idx + W_size].reshape(W_shape)
        idx += W_size
        b = theta[idx: idx + b_size].reshape(b_shape)
        idx += b_size
        params.append({"W": W, "b": b})
    return params

# ============================================================
# UTILITIES
# ============================================================
def segment_sum(values, segment_ids, num_segments):
    out = jnp.zeros((num_segments,), dtype=values.dtype)
    return out.at[segment_ids].add(values)

# ============================================================
# STRATIFIED SAMPLING
# ============================================================
def sample_stratified_3d(key, x0, x1, y0, y1, t0, t1, NX, NY, NT):
    dx = (DTYPE(x1) - DTYPE(x0)) / DTYPE(NX)
    dy = (DTYPE(y1) - DTYPE(y0)) / DTYPE(NY)
    dt = (DTYPE(t1) - DTYPE(t0)) / DTYPE(NT)

    it, iy, ix = jnp.meshgrid(jnp.arange(NT), jnp.arange(NY), jnp.arange(NX), indexing="ij")
    it = it.reshape(-1).astype(jnp.int32)
    iy = iy.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)
    ids = (it * (NY * NX) + iy * NX + ix).astype(jnp.int32)

    xb = DTYPE(x0) + DTYPE(ix) * dx
    yb = DTYPE(y0) + DTYPE(iy) * dy
    tb = DTYPE(t0) + DTYPE(it) * dt

    K = NX * NY * NT
    u = random.uniform(key, (K, 3), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = xb + u[:, 0] * dx
    ys = yb + u[:, 1] * dy
    ts = tb + u[:, 2] * dt
    X = jnp.stack([xs, ys, ts], axis=1)
    return X, ids

def sample_bc_time_binned(key, K, *, x_fixed=None, y_fixed=None, t0=t_min, t1=t_max,
                          x_lo=None, x_hi=None, y_lo=None, y_hi=None):
    dt = (DTYPE(t1) - DTYPE(t0)) / DTYPE(K)
    j = jnp.arange(K, dtype=jnp.int32)
    tb = DTYPE(t0) + DTYPE(j) * dt

    key, kt, kr = random.split(key, 3)
    u_t = random.uniform(kt, (K, N_BC_PER_BIN), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (tb[:, None] + u_t * dt).reshape(-1, 1)
    ids = jnp.repeat(j, N_BC_PER_BIN)

    if x_fixed is not None:
        lo = DTYPE(y_min if y_lo is None else y_lo)
        hi = DTYPE(y_max if y_hi is None else y_hi)
        y = random.uniform(kr, (ts.shape[0], 1), minval=lo, maxval=hi, dtype=DTYPE)
        x = DTYPE(x_fixed) * jnp.ones_like(y)
        return jnp.concatenate([x, y, ts], axis=1), ids

    if y_fixed is not None:
        lo = DTYPE(x_min if x_lo is None else x_lo)
        hi = DTYPE(x_max if x_hi is None else x_hi)
        x = random.uniform(kr, (ts.shape[0], 1), minval=lo, maxval=hi, dtype=DTYPE)
        y = DTYPE(y_fixed) * jnp.ones_like(x)
        return jnp.concatenate([x, y, ts], axis=1), ids

    raise ValueError("Provide x_fixed or y_fixed")

def sample_ic_xy(key):
    y0 = DTYPE(0.02)
    y1 = y_max - DTYPE(0.02)

    dx = (x_max - x_min) / DTYPE(NX_IC)
    dy = (y1 - y0) / DTYPE(NY_IC)

    iy, ix = jnp.meshgrid(jnp.arange(NY_IC), jnp.arange(NX_IC), indexing="ij")
    iy = iy.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)
    ids0 = (iy * NX_IC + ix).astype(jnp.int32)

    xb = x_min + DTYPE(ix) * dx
    yb = y0 + DTYPE(iy) * dy

    u = random.uniform(key, (K_IC, N_IC_PER_BIN, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = (xb[:, None] + u[:, :, 0] * dx).reshape(-1, 1)
    ys = (yb[:, None] + u[:, :, 1] * dy).reshape(-1, 1)
    ts = t_min * jnp.ones_like(xs)

    X = jnp.concatenate([xs, ys, ts], axis=1)
    ids = jnp.repeat(ids0, N_IC_PER_BIN)
    return X, ids

def sample_interface_yt(key):
    dy = (y_max - y_min) / DTYPE(NY_IF)
    dt = (t_max - t_min) / DTYPE(NT_IF)

    it, iy = jnp.meshgrid(jnp.arange(NT_IF), jnp.arange(NY_IF), indexing="ij")
    it = it.reshape(-1).astype(jnp.int32)
    iy = iy.reshape(-1).astype(jnp.int32)
    ids0 = (it * NY_IF + iy).astype(jnp.int32)

    yb = y_min + DTYPE(iy) * dy
    tb = t_min + DTYPE(it) * dt

    u = random.uniform(key, (NY_IF * NT_IF, N_IF_PER_BIN, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    ys = (yb[:, None] + u[:, :, 0] * dy).reshape(-1, 1)
    ts = (tb[:, None] + u[:, :, 1] * dt).reshape(-1, 1)
    xs = Ls * jnp.ones_like(ys)

    X = jnp.concatenate([xs, ys, ts], axis=1)
    ids = jnp.repeat(ids0, N_IF_PER_BIN)
    return X, ids

# ============================================================
# JITTER + PROJECTIONS
# ============================================================
def jitter_xyt(key, X, sx, sy, st, clip_k=2.0):
    if X.shape[0] == 0 or ((sx <= 0) and (sy <= 0) and (st <= 0)):
        return X

    key, kx, ky, kt = random.split(key, 4)
    dx = DTYPE(sx) * random.normal(kx, (X.shape[0],), dtype=DTYPE)
    dy = DTYPE(sy) * random.normal(ky, (X.shape[0],), dtype=DTYPE)
    dt = DTYPE(st) * random.normal(kt, (X.shape[0],), dtype=DTYPE)

    k = DTYPE(clip_k)
    dx = jnp.clip(dx, -k * DTYPE(sx), k * DTYPE(sx))
    dy = jnp.clip(dy, -k * DTYPE(sy), k * DTYPE(sy))
    dt = jnp.clip(dt, -k * DTYPE(st), k * DTYPE(st))

    x = jnp.clip(X[:, 0] + dx, x_min, x_max)
    y = jnp.clip(X[:, 1] + dy, y_min, y_max)
    t = jnp.clip(X[:, 2] + dt, t_min, t_max)
    return jnp.stack([x, y, t], axis=1)

def project_x_fixed(X, x_fixed):
    if X.shape[0] == 0:
        return X
    return jnp.stack([
        DTYPE(x_fixed) * jnp.ones((X.shape[0],), dtype=DTYPE),
        X[:, 1],
        X[:, 2],
    ], axis=1)

def project_y_fixed(X, y_fixed, *, x_lo=None, x_hi=None):
    if X.shape[0] == 0:
        return X
    x = X[:, 0]
    if (x_lo is not None) or (x_hi is not None):
        lo = DTYPE(x_lo if x_lo is not None else x_min)
        hi = DTYPE(x_hi if x_hi is not None else x_max)
        x = jnp.clip(x, lo, hi)
    y = DTYPE(y_fixed) * jnp.ones((X.shape[0],), dtype=DTYPE)
    return jnp.stack([x, y, X[:, 2]], axis=1)

def project_t0(X):
    if X.shape[0] == 0:
        return X
    return jnp.stack([
        X[:, 0],
        X[:, 1],
        t_min * jnp.ones((X.shape[0],), dtype=DTYPE),
    ], axis=1)

def project_interface(X):
    return project_x_fixed(X, Ls)

# ============================================================
# PDE RESIDUALS
# ============================================================
def forward6(params, xyt):
    return mlp_apply(params, xyt[None, :])[0, :]

def eval_fields_and_derivs(params, X):
    out = vmap(lambda z: forward6(params, z))(X)
    J   = vmap(jacrev(lambda z: forward6(params, z)))(X)
    H   = vmap(hessian(lambda z: forward6(params, z)))(X)
    return out, J, H

def residuals_all(params, X):
    out, J, H = eval_fields_and_derivs(params, X)

    phi = out[:, 0]
    Ts  = out[:, 1]
    u   = out[:, 2]
    v   = out[:, 3]
    p   = out[:, 4]
    Tf  = out[:, 5]

    x = X[:, 0]
    is_solid = (x <= Ls)

    phi_t  = J[:, 0, 2]
    phi_xx = H[:, 0, 0, 0]
    phi_yy = H[:, 0, 1, 1]
    lap_phi = phi_xx + phi_yy

    Ts_x  = J[:, 1, 0]
    Ts_y  = J[:, 1, 1]
    Ts_t  = J[:, 1, 2]
    Ts_xx = H[:, 1, 0, 0]
    Ts_yy = H[:, 1, 1, 1]

    u_x, u_y, u_t = J[:, 2, 0], J[:, 2, 1], J[:, 2, 2]
    v_x, v_y, v_t = J[:, 3, 0], J[:, 3, 1], J[:, 3, 2]
    p_x, p_y      = J[:, 4, 0], J[:, 4, 1]
    Tf_x, Tf_y, Tf_t = J[:, 5, 0], J[:, 5, 1], J[:, 5, 2]

    u_xx, u_yy   = H[:, 2, 0, 0], H[:, 2, 1, 1]
    v_xx, v_yy   = H[:, 3, 0, 0], H[:, 3, 1, 1]
    Tf_xx, Tf_yy = H[:, 5, 0, 0], H[:, 5, 1, 1]

    T_for_sigma = jnp.where(is_solid, Ts, Tf)
    D_here      = jnp.where(is_solid, D_fuel, D_fluid)
    sig_here    = jnp.where(is_solid, sigma_af_fuel(T_for_sigma), sigma_af_fluid(T_for_sigma))

    r_phi_raw = (DTYPE(1.0) / v_neu) * phi_t - D_here * lap_phi - sig_here * phi
    r_phi = r_phi_raw / PHI_RES_SCALE

    div_k_grad_Ts = div_k_grad_scalar(k_s_fun(Ts), dk_s_dT, Ts, Ts_x, Ts_y, Ts_xx, Ts_yy)
    r_Ts_raw = rho_s * cp_s_fun(Ts) * Ts_t - div_k_grad_Ts - POWER_COEF * phi
    r_Ts = r_Ts_raw / TS_RES_SCALE

    r_cont_raw = u_x + v_y
    r_cont = r_cont_raw / CONT_SCALE

    mu_here = mu_f_fun(Tf)
    visc_u = div_mu_grad_component(mu_here, dmu_f_dT, Tf, Tf_x, Tf_y, u_x, u_y, u_xx, u_yy)
    visc_v = div_mu_grad_component(mu_here, dmu_f_dT, Tf, Tf_x, Tf_y, v_x, v_y, v_xx, v_yy)

    r_u_raw = rho_f * (u_t + u * u_x + v * u_y) + p_x - visc_u
    r_v_raw = rho_f * (v_t + u * v_x + v * v_y) + p_y - visc_v

    cp_here = cp_f_fun(Tf)
    div_k_grad_Tf = div_k_grad_scalar(k_f_fun(Tf), dk_f_dT, Tf, Tf_x, Tf_y, Tf_xx, Tf_yy)
    r_Tf_raw = rho_f * cp_here * (Tf_t + u * Tf_x + v * Tf_y) - div_k_grad_Tf

    r_u  = r_u_raw / MOMX_RES_SCALE
    r_v  = r_v_raw / MOMY_RES_SCALE
    r_Tf = r_Tf_raw / TF_RES_SCALE

    return r_phi, r_Ts, r_cont, r_u, r_v, r_Tf

# ============================================================
# OBJECTIVE
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def F_and_g_obj(theta, shapes, X_obj_phi, X_obj_s, X_obj_f,
                w_obj_phi, w_obj_Ts, w_obj_cont, w_obj_ru, w_obj_rv, w_obj_rTf):
    def obj_theta(th):
        params = unflatten_params(th, shapes)

        r_phi_obj, _, _, _, _, _ = residuals_all(params, X_obj_phi)
        _, r_Ts_obj, _, _, _, _  = residuals_all(params, X_obj_s)
        _, _, r_cont_obj, r_u_obj, r_v_obj, r_Tf_obj = residuals_all(params, X_obj_f)

        return (
            w_obj_phi  * jnp.mean(r_phi_obj ** 2)
            + w_obj_Ts * jnp.mean(r_Ts_obj ** 2)
            + w_obj_cont * jnp.mean(r_cont_obj ** 2)
            + w_obj_ru   * jnp.mean(r_u_obj ** 2)
            + w_obj_rv   * jnp.mean(r_v_obj ** 2)
            + w_obj_rTf  * jnp.mean(r_Tf_obj ** 2)
        )

    val, g = jax.value_and_grad(obj_theta)(theta)
    return val, g

# ============================================================
# CONSTRAINT VECTOR
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def constraint_vector(theta, shapes,
                      X_con_phi, ids_con_phi,
                      X_con_s, ids_con_s,
                      X_con_f, ids_con_f,
                      X_phi_left, ids_phi_left,
                      X_phi_right, ids_phi_right,
                      X_phi_y0, ids_phi_y0,
                      X_phi_y1, ids_phi_y1,
                      X_Ts_x0, ids_Ts_x0,
                      X_f_inlet, ids_f_inlet,
                      X_f_outlet, ids_f_outlet,
                      X_f_right, ids_f_right,
                      X_if, ids_if,
                      X_ic, ids_ic):
    params = unflatten_params(theta, shapes)

    # PDE hard constraints
    r_phi, _, _, _, _, _ = residuals_all(params, X_con_phi)
    _, r_Ts, _, _, _, _  = residuals_all(params, X_con_s)
    _, _, r_cont, r_u, r_v, r_Tf = residuals_all(params, X_con_f)

    c_pde_phi  = segment_sum(r_phi,  ids_con_phi, K_CON_PHI) / DTYPE(N_PER_CELL)
    c_pde_Ts   = segment_sum(r_Ts,   ids_con_s,   K_CON_S)   / DTYPE(N_PER_CELL)
    c_pde_cont = segment_sum(r_cont, ids_con_f,   K_CON_F)   / DTYPE(N_PER_CELL)
    c_pde_ru   = segment_sum(r_u,    ids_con_f,   K_CON_F)   / DTYPE(N_PER_CELL)
    c_pde_rv   = segment_sum(r_v,    ids_con_f,   K_CON_F)   / DTYPE(N_PER_CELL)
    c_pde_rTf  = segment_sum(r_Tf,   ids_con_f,   K_CON_F)   / DTYPE(N_PER_CELL)

    # phi BCs
    phiL = mlp_apply(params, X_phi_left)[:, 0]
    yL, tL = X_phi_left[:, 1], X_phi_left[:, 2]
    c_phi_left = segment_sum((phiL - phi_left_target(yL, tL)) / PHI_VAL_SCALE, ids_phi_left, K_BC_PHI_LEFT) / DTYPE(N_BC_PER_BIN)

    phiR = mlp_apply(params, X_phi_right)[:, 0]
    c_phi_right = segment_sum((phiR - PHI_RIGHT_VAL) / PHI_VAL_SCALE, ids_phi_right, K_BC_PHI_RIGHT) / DTYPE(N_BC_PER_BIN)

    phiB = mlp_apply(params, X_phi_y0)[:, 0]
    c_phi_y0 = segment_sum((phiB - PHI_TOPBOT_VAL) / PHI_VAL_SCALE, ids_phi_y0, K_BC_PHI_Y0) / DTYPE(N_BC_PER_BIN)

    phiT = mlp_apply(params, X_phi_y1)[:, 0]
    c_phi_y1 = segment_sum((phiT - PHI_TOPBOT_VAL) / PHI_VAL_SCALE, ids_phi_y1, K_BC_PHI_Y1) / DTYPE(N_BC_PER_BIN)

    c_bc_phi = jnp.concatenate([c_phi_left, c_phi_right, c_phi_y0, c_phi_y1], axis=0)

    # solid BC
    def Ts_fun(z):
        return forward6(params, z)[1]

    Tsx_x0 = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_Ts_x0)
    c_bc_sol = segment_sum(Tsx_x0 / GRAD_TSX_SCALE, ids_Ts_x0, K_BC_TS_X0) / DTYPE(N_BC_PER_BIN)

    # fluid BCs
    out_in = mlp_apply(params, X_f_inlet)
    u_in, v_in, Tf_in = out_in[:, 2], out_in[:, 3], out_in[:, 5]
    x_in = X_f_inlet[:, 0]
    v_tar = inlet_v_profile(x_in)

    c_in_u  = segment_sum((u_in - DTYPE(0.0)) / U_VAL_SCALE, ids_f_inlet, K_BC_F_INLET) / DTYPE(N_BC_PER_BIN)
    c_in_v  = segment_sum((v_in - v_tar) / V_VAL_SCALE,      ids_f_inlet, K_BC_F_INLET) / DTYPE(N_BC_PER_BIN)
    c_in_Tf = segment_sum((Tf_in - T_INLET) / TF_VAL_SCALE,  ids_f_inlet, K_BC_F_INLET) / DTYPE(N_BC_PER_BIN)

    p_out = mlp_apply(params, X_f_outlet)[:, 4]
    c_out_p = segment_sum((p_out - P_OUTLET) / P_VAL_SCALE, ids_f_outlet, K_BC_F_OUTLET) / DTYPE(N_BC_PER_BIN)

    out_right = mlp_apply(params, X_f_right)
    u_right  = out_right[:, 2]
    v_right  = out_right[:, 3]
    p_right  = out_right[:, 4]
    Tf_right = out_right[:, 5]

    y_r = X_f_right[:, 1]
    t_r = X_f_right[:, 2]

    u_tar_right  = u_right_target(y_r, t_r)
    v_tar_right  = v_right_target(y_r, t_r)
    p_tar_right  = p_right_target(y_r, t_r)
    Tf_tar_right = Tf_right_target(y_r, t_r)

    c_r_u  = segment_sum((u_right  - u_tar_right)  / U_VAL_SCALE,  ids_f_right, K_BC_F_RIGHT) / DTYPE(N_BC_PER_BIN)
    c_r_v  = segment_sum((v_right  - v_tar_right)  / V_VAL_SCALE,  ids_f_right, K_BC_F_RIGHT) / DTYPE(N_BC_PER_BIN)
    c_r_p  = segment_sum((p_right  - p_tar_right)  / P_VAL_SCALE,  ids_f_right, K_BC_F_RIGHT) / DTYPE(N_BC_PER_BIN)
    c_r_Tf = segment_sum((Tf_right - Tf_tar_right) / TF_VAL_SCALE, ids_f_right, K_BC_F_RIGHT) / DTYPE(N_BC_PER_BIN)

    c_bc_fld = jnp.concatenate([
        c_in_u, c_in_v, c_in_Tf,
        c_out_p,
        c_r_u, c_r_v, c_r_p, c_r_Tf
    ], axis=0)

    # interface
    out_if = mlp_apply(params, X_if)
    Ts_if, Tf_if = out_if[:, 1], out_if[:, 5]

    c_if_T = segment_sum((Ts_if - Tf_if) / T_IF_VAL_SCALE, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)

    Tsx_if = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_if)

    def Tf_fun(z):
        return forward6(params, z)[5]

    Tfx_if = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_if)

    flux_jump = -k_s_fun(Ts_if) * Tsx_if + k_f_fun(Tf_if) * Tfx_if
    c_if_q = segment_sum(flux_jump / FLUX_SCALE, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)

    c_if = jnp.concatenate([c_if_T, c_if_q], axis=0)

    # IC
    out0 = mlp_apply(params, X_ic)
    phi0 = out0[:, 0]
    Ts0  = out0[:, 1]
    u0   = out0[:, 2]
    v0   = out0[:, 3]
    Tf0  = out0[:, 5]

    x0 = X_ic[:, 0]
    y0 = X_ic[:, 1]

    m_s0 = (x0 <= Ls).astype(DTYPE)
    m_f0 = (x0 >  Ls).astype(DTYPE)

    c_ic_phi = segment_sum((phi0 - phi_ic(y0)) / PHI_VAL_SCALE, ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)
    c_ic_Ts  = segment_sum(m_s0 * ((Ts0 - T_INLET) / TS_VAL_SCALE), ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)
    c_ic_u   = segment_sum(m_f0 * ((u0 - FLUID_U_IC) / U_VAL_SCALE), ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)
    c_ic_v   = segment_sum(m_f0 * ((v0 - FLUID_V_IC) / V_VAL_SCALE), ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)
    c_ic_Tf  = segment_sum(m_f0 * ((Tf0 - FLUID_T_IC) / TF_VAL_SCALE), ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)

    c_ic = jnp.concatenate([c_ic_phi, c_ic_Ts, c_ic_u, c_ic_v, c_ic_Tf], axis=0)

    return jnp.concatenate([
        c_pde_phi,
        c_pde_Ts,
        c_pde_cont, c_pde_ru, c_pde_rv, c_pde_rTf,
        c_bc_phi,
        c_bc_sol,
        c_bc_fld,
        c_if,
        c_ic
    ], axis=0)

@partial(jax.jit, static_argnames=("shapes",))
def C_and_J(theta, shapes, *args):
    def c_fun(th):
        return constraint_vector(th, shapes, *args)

    def c_fun_aux(th):
        c = c_fun(th)
        return c, c

    J, c = jax.jacrev(c_fun_aux, has_aux=True)(theta)
    return c, J

@partial(jax.jit, static_argnames=("shapes",))
def C_only(theta, shapes, *args):
    return constraint_vector(theta, shapes, *args)

# ============================================================
# KKT solve
# ============================================================
@jax.jit
def kkt_solve_once(H, Jac, Grad, Cons, mu_damp):
    n = H.shape[0]
    m = Cons.shape[0]
    I_m = jnp.eye(m, dtype=H.dtype)
    top = jnp.concatenate([H, Jac.T], axis=1)
    bottom = jnp.concatenate([Jac, -mu_damp * I_m], axis=1)
    KKT = jnp.concatenate([top, bottom], axis=0)
    rhs = -jnp.concatenate([Grad, Cons])
    sol = jnp.linalg.solve(KKT, rhs)
    d = sol[:n]
    y = sol[n:]
    return d, y

# ============================================================
# STEP HELPERS
# ============================================================
def cal_tau_mu(H, d, sigma, tau_pre, eps_tau, g, c, mu_eff, y):
    denom = float(g @ d + 0.5 * (d @ (H @ d)))
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(mu_eff * y, 1))
    delta_c = c1 - mu_y1
    if (denom <= 1e-12) or (delta_c <= 0.0):
        tau_trial = float("inf")
    else:
        tau_trial = (1.0 - sigma) * delta_c / denom
    if tau_pre <= tau_trial:
        return tau_pre
    return min(tau_trial, (1.0 - eps_tau) * tau_pre)

def cal_ksi_mu(d, tau, ksi_old, eps_ksi, g, c, mu_eff, y):
    d2 = float(jnp.linalg.norm(d) ** 2)
    if d2 <= 1e-18 or tau <= 1e-18:
        return ksi_old
    gd = float(g @ d)
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(mu_eff * y, 1))
    delta_c = c1 - mu_y1
    Dl = -tau * gd + delta_c
    ksi_trial = Dl / (tau * d2)
    ksi_trial = max(0.0, ksi_trial)
    if ksi_old <= ksi_trial:
        return ksi_old
    return min(ksi_trial, (1.0 - eps_ksi) * ksi_old)

def phi_mu(alpha, eta, beta, tau, g, d, c, mu_eff, y, L, Gamma):
    gd = float(g @ d)
    d2 = float(d @ d)
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(mu_eff * y, 1))
    delta_c = c1 - mu_y1
    Dl = -tau * gd + delta_c
    term1 = (eta - 1.0) * alpha * beta * Dl
    term2 = (abs(1.0 - alpha) - 1.0 + alpha) * c1
    term3 = 0.5 * (tau * L + Gamma) * (alpha ** 2) * d2
    return term1 + term2 + term3

def cal_alpha_mu(d, eta, beta, ksi, tau, L, Gamma, theta_val, g, c, mu_eff, y):
    denom = (tau * L + Gamma)
    if denom <= 1e-12:
        return 0.0
    alpha_min = 2.0 * (1.0 - eta) * beta * ksi * tau / denom
    a = max(alpha_min, 0.0)
    while (
        phi_mu(1.1 * a, eta, beta, tau, g, d, c, mu_eff, y, L, Gamma) < 0.0
        and (1.1 * a < alpha_min + theta_val * beta)
    ):
        a *= 1.1
    return float(a)

# ============================================================
# BLOCK SLICES / ROW SCALING
# ============================================================
def _block_slices():
    s = {}
    i = 0

    s["pde_phi"]  = slice(i, i + K_CON_PHI); i += K_CON_PHI
    s["pde_Ts"]   = slice(i, i + K_CON_S);   i += K_CON_S
    s["pde_cont"] = slice(i, i + K_CON_F);   i += K_CON_F
    s["pde_ru"]   = slice(i, i + K_CON_F);   i += K_CON_F
    s["pde_rv"]   = slice(i, i + K_CON_F);   i += K_CON_F
    s["pde_rTf"]  = slice(i, i + K_CON_F);   i += K_CON_F

    s["bc_phi"]   = slice(i, i + N_BC_PHI); i += N_BC_PHI
    s["bc_solid"] = slice(i, i + N_BC_SOL); i += N_BC_SOL
    s["bc_fluid"] = slice(i, i + N_BC_FLD); i += N_BC_FLD

    s["iface"]    = slice(i, i + N_IF); i += N_IF
    s["ic"]       = slice(i, i + N_IC); i += N_IC

    assert i == M_CON, (i, M_CON)
    return s

BLOCKS = _block_slices()

S_TARGET_PDE = DTYPE(1.0)
S_TARGET_BC  = DTYPE(1.0)
S_TARGET_IF  = DTYPE(1.0)
S_TARGET_IC  = DTYPE(1.0)
S_ROW_MIN    = DTYPE(1e-6)
S_ROW_MAX    = DTYPE(1.0)

@jax.jit
def row_rms(J_rows):
    return jnp.sqrt(jnp.mean(J_rows * J_rows, axis=1) + EPS)

@jax.jit
def make_block_row_scales(J, target_pde, target_bc, target_if, target_ic):
    s = jnp.ones((M_CON,), dtype=DTYPE)

    def set_block(name, target):
        sl = BLOCKS[name]
        rr = row_rms(J[sl, :])
        sb = target / jnp.maximum(target, rr)
        sb = jnp.clip(sb, S_ROW_MIN, S_ROW_MAX)
        return rr, sb

    rr_phi, sb_phi   = set_block("pde_phi", target_pde)
    rr_Ts, sb_Ts     = set_block("pde_Ts", target_pde)
    rr_cont, sb_cont = set_block("pde_cont", target_pde)
    rr_ru, sb_ru     = set_block("pde_ru", target_pde)
    rr_rv, sb_rv     = set_block("pde_rv", target_pde)
    rr_rTf, sb_rTf   = set_block("pde_rTf", target_pde)

    s = s.at[BLOCKS["pde_phi"]].set(sb_phi)
    s = s.at[BLOCKS["pde_Ts"]].set(sb_Ts)
    s = s.at[BLOCKS["pde_cont"]].set(sb_cont)
    s = s.at[BLOCKS["pde_ru"]].set(sb_ru)
    s = s.at[BLOCKS["pde_rv"]].set(sb_rv)
    s = s.at[BLOCKS["pde_rTf"]].set(sb_rTf)

    rr_bcphi, sb_bcphi = set_block("bc_phi", target_bc)
    rr_bcsol, sb_bcsol = set_block("bc_solid", target_bc)
    rr_bcfld, sb_bcfld = set_block("bc_fluid", target_bc)

    s = s.at[BLOCKS["bc_phi"]].set(sb_bcphi)
    s = s.at[BLOCKS["bc_solid"]].set(sb_bcsol)
    s = s.at[BLOCKS["bc_fluid"]].set(sb_bcfld)

    rr_if, sb_if = set_block("iface", target_if)
    rr_ic, sb_ic = set_block("ic", target_ic)

    s = s.at[BLOCKS["iface"]].set(sb_if)
    s = s.at[BLOCKS["ic"]].set(sb_ic)

    return s

@jax.jit
def apply_row_scale(c, J, s_row):
    return s_row * c, s_row[:, None] * J

def _rms(x):
    return float(jnp.sqrt(jnp.mean(x * x) + 1e-30))

def _maxabs(x):
    return float(jnp.max(jnp.abs(x)))

def print_block_stats(prefix, c):
    def pr(name):
        sl = BLOCKS[name]
        v = c[sl]
        print(f"  {name:10s}: rms={_rms(v):.3e}  max={_maxabs(v):.3e}")
    print(prefix)
    pr("pde_phi")
    pr("pde_Ts")
    pr("pde_cont")
    pr("pde_ru")
    pr("pde_rv")
    pr("pde_rTf")
    pr("bc_phi")
    pr("bc_solid")
    pr("bc_fluid")
    pr("iface")
    pr("ic")

# ============================================================
# TRAIN SQP
# ============================================================
def train_sqp(
    seed=0,
    hidden_dim=30,
    num_hidden=3,
    max_iters=1000,
    print_every=10,
    beta_shift=400.0,
    beta_power=0.6,
    alpha_cap=1e1,
    L_lip=60.0,
    Gamma_lip=60.0,
    jitter_every_iter=True,
    sx=1e-7,
    sy=1e-6,
    st=1e-6,
    sx_ic=1e-7,
    sy_ic=1e-6,
    sy_if=1e-6,
    st_if=1e-6,
    w_obj_phi=20.0,
    w_obj_Ts=10.0,
    w_obj_cont=30.0,
    w_obj_ru=10.0,
    w_obj_rv=10.0,
    w_obj_rTf=10.0,
):
    key = random.PRNGKey(seed)

    layer_sizes = [3] + [hidden_dim] * num_hidden + [6]
    key, k0 = random.split(key)
    params0 = init_mlp_params(k0, layer_sizes)
    theta, shapes = flatten_params(params0)

    n = theta.shape[0]
    H = jnp.eye(n, dtype=DTYPE)

    # objective sets
    key, k_obj_phi, k_obj_s, k_obj_f = random.split(key, 4)
    X_obj_phi, _ = sample_stratified_3d(k_obj_phi, x_min, x_max, y_min, y_max, t_min, t_max, NX_OBJ_PHI, NY_OBJ_PHI, NT_OBJ_PHI)
    X_obj_s, _   = sample_stratified_3d(k_obj_s, x_min, Ls, y_min, y_max, t_min, t_max, NX_OBJ_S, NY_OBJ_S, NT_OBJ_S)
    X_obj_f, _   = sample_stratified_3d(k_obj_f, Ls, x_max, y_min, y_max, t_min, t_max, NX_OBJ_F, NY_OBJ_F, NT_OBJ_F)

    # constraint sets
    key, k_con_phi, k_con_s, k_con_f = random.split(key, 4)
    X_con_phi, ids_con_phi = sample_stratified_3d(k_con_phi, x_min, x_max, y_min, y_max, t_min, t_max, NX_CON_PHI, NY_CON_PHI, NT_CON_PHI)
    X_con_s, ids_con_s     = sample_stratified_3d(k_con_s,   x_min, Ls,    y_min, y_max, t_min, t_max, NX_CON_S,   NY_CON_S,   NT_CON_S)
    X_con_f, ids_con_f     = sample_stratified_3d(k_con_f,   Ls,    x_max, y_min, y_max, t_min, t_max, NX_CON_F,   NY_CON_F,   NT_CON_F)

    key, k1, k2, k3, k4, k5, k6, k7, k8, k9, k_ic = random.split(key, 11)

    X_phi_left,  ids_phi_left  = sample_bc_time_binned(k1, K_BC_PHI_LEFT,  x_fixed=x_min, y_lo=y_min, y_hi=y_max)
    X_phi_right, ids_phi_right = sample_bc_time_binned(k2, K_BC_PHI_RIGHT, x_fixed=x_max, y_lo=y_min, y_hi=y_max)
    X_phi_y0,    ids_phi_y0    = sample_bc_time_binned(k3, K_BC_PHI_Y0,    y_fixed=y_min, x_lo=x_min, x_hi=x_max)
    X_phi_y1,    ids_phi_y1    = sample_bc_time_binned(k4, K_BC_PHI_Y1,    y_fixed=y_max, x_lo=x_min, x_hi=x_max)

    X_Ts_x0, ids_Ts_x0 = sample_bc_time_binned(k5, K_BC_TS_X0, x_fixed=x_min, y_lo=y_min, y_hi=y_max)

    xL = float(Ls + X_MARGIN)
    xR = float(x_max - X_MARGIN)

    X_f_inlet,  ids_f_inlet  = sample_bc_time_binned(k6, K_BC_F_INLET,  y_fixed=y_min, x_lo=xL, x_hi=xR)
    X_f_outlet, ids_f_outlet = sample_bc_time_binned(k7, K_BC_F_OUTLET, y_fixed=y_max, x_lo=xL, x_hi=xR)
    X_f_right,  ids_f_right  = sample_bc_time_binned(k8, K_BC_F_RIGHT,  x_fixed=x_max, y_lo=y_min, y_hi=y_max)

    X_if, ids_if = sample_interface_yt(k9)
    X_ic, ids_ic = sample_ic_xy(k_ic)

    eta, sigma = 0.25, 0.1
    eps_tau, eps_ksi = 1e-2, 1e-2
    theta_val = 10.0
    tau_k, ksi_k = 1.0, 1.0

    print("M_CON =", int(M_CON), "n_params =", int(n))

    s_row_anchor = None
    t0_clock = time.time()

    for it in range(1, max_iters + 1):
        k_beta = (it // 10) * 10
        beta_k = float(min(1.0, (beta_shift / (beta_shift + k_beta)) ** beta_power))

        if jitter_every_iter:
            key, *ks = random.split(key, 16)
            (kj_obj_phi, kj_obj_s, kj_obj_f,
             kj_con_phi, kj_con_s, kj_con_f,
             kjL, kjR, kjB, kjT,
             kjSx, kjIn, kjOut, kjRight, kjIf) = ks[:15]
            key, kjIc = random.split(key)

            X_obj_phi_use = jitter_xyt(kj_obj_phi, X_obj_phi, sx, sy, st)
            X_obj_s_use   = jitter_xyt(kj_obj_s,   X_obj_s,   sx, sy, st)
            X_obj_f_use   = jitter_xyt(kj_obj_f,   X_obj_f,   sx, sy, st)

            X_con_phi_use = jitter_xyt(kj_con_phi, X_con_phi, sx, sy, st)
            X_con_s_use   = jitter_xyt(kj_con_s,   X_con_s,   sx, sy, st)
            X_con_f_use   = jitter_xyt(kj_con_f,   X_con_f,   sx, sy, st)

            X_phi_left_use  = project_x_fixed(jitter_xyt(kjL, X_phi_left,  0.0, sy, st), x_min)
            X_phi_right_use = project_x_fixed(jitter_xyt(kjR, X_phi_right, 0.0, sy, st), x_max)
            X_phi_y0_use    = project_y_fixed(jitter_xyt(kjB, X_phi_y0, sx, 0.0, st), y_min)
            X_phi_y1_use    = project_y_fixed(jitter_xyt(kjT, X_phi_y1, sx, 0.0, st), y_max)

            X_Ts_x0_use = project_x_fixed(jitter_xyt(kjSx, X_Ts_x0, 0.0, sy, st), x_min)

            X_f_inlet_use  = project_y_fixed(jitter_xyt(kjIn,    X_f_inlet,  sx, 0.0, st), y_min, x_lo=xL, x_hi=xR)
            X_f_outlet_use = project_y_fixed(jitter_xyt(kjOut,   X_f_outlet, sx, 0.0, st), y_max, x_lo=xL, x_hi=xR)
            X_f_right_use  = project_x_fixed(jitter_xyt(kjRight, X_f_right,  0.0, sy, st), x_max)

            X_if_use = project_interface(jitter_xyt(kjIf, X_if, 0.0, sy_if, st_if))
            X_ic_use = project_t0(jitter_xyt(kjIc, X_ic, sx_ic, sy_ic, 0.0))
        else:
            X_obj_phi_use = X_obj_phi
            X_obj_s_use   = X_obj_s
            X_obj_f_use   = X_obj_f
            X_con_phi_use = X_con_phi
            X_con_s_use   = X_con_s
            X_con_f_use   = X_con_f
            X_phi_left_use, X_phi_right_use, X_phi_y0_use, X_phi_y1_use = X_phi_left, X_phi_right, X_phi_y0, X_phi_y1
            X_Ts_x0_use = X_Ts_x0
            X_f_inlet_use, X_f_outlet_use, X_f_right_use = X_f_inlet, X_f_outlet, X_f_right
            X_if_use, X_ic_use = X_if, X_ic

        obj_val, g = F_and_g_obj(
            theta, shapes,
            X_obj_phi_use, X_obj_s_use, X_obj_f_use,
            DTYPE(w_obj_phi), DTYPE(w_obj_Ts), DTYPE(w_obj_cont),
            DTYPE(w_obj_ru), DTYPE(w_obj_rv), DTYPE(w_obj_rTf)
        )

        args = (
            X_con_phi_use, ids_con_phi,
            X_con_s_use, ids_con_s,
            X_con_f_use, ids_con_f,
            X_phi_left_use, ids_phi_left,
            X_phi_right_use, ids_phi_right,
            X_phi_y0_use, ids_phi_y0,
            X_phi_y1_use, ids_phi_y1,
            X_Ts_x0_use, ids_Ts_x0,
            X_f_inlet_use, ids_f_inlet,
            X_f_outlet_use, ids_f_outlet,
            X_f_right_use, ids_f_right,
            X_if_use, ids_if,
            X_ic_use, ids_ic,
        )

        c, J = C_and_J(theta, shapes, *args)

        if s_row_anchor is None:
            s_row_anchor = make_block_row_scales(J, S_TARGET_PDE, S_TARGET_BC, S_TARGET_IF, S_TARGET_IC)

        c_s, J_s = apply_row_scale(c, J, s_row_anchor)

        d, y_s = kkt_solve_once(H, J_s, g, c_s, MU_DAMP_FIXED)
        y = s_row_anchor * y_s

        dn = float(jnp.linalg.norm(d, jnp.inf))
        if dn > 1e-12:
            tau_k = cal_tau_mu(H, d, sigma, tau_k, eps_tau, g, c_s, MU_DAMP_FIXED, y_s)
            ksi_k = cal_ksi_mu(d, tau_k, ksi_k, eps_ksi, g, c_s, MU_DAMP_FIXED, y_s)
            alpha = cal_alpha_mu(d, eta, beta_k, ksi_k, tau_k, L_lip, Gamma_lip, theta_val, g, c_s, MU_DAMP_FIXED, y_s)
        else:
            alpha = 0.0

        alpha = float(min(alpha, alpha_cap))
        theta = theta + DTYPE(alpha) * d

        if it % int(print_every) == 0:
            c_now = C_only(theta, shapes, *args)
            feas_raw = float(jnp.mean(c_now ** 2))
            station  = float(jnp.linalg.norm(g + J.T @ y, jnp.inf))

            print(f"[it={it}] obj={float(obj_val):.3e} feas_raw={feas_raw:.3e} alpha={alpha:.2e} beta={beta_k:.2e} station={station:.2e} dn={dn:.2e}")
            print_block_stats("blocks (unscaled c):", c_now)

            params_now = unflatten_params(theta, shapes)
            r_phi_obj, _, _, _, _, _ = residuals_all(params_now, X_obj_phi_use)
            _, r_Ts_obj, _, _, _, _  = residuals_all(params_now, X_obj_s_use)
            _, _, r_cont_obj, r_u_obj, r_v_obj, r_Tf_obj = residuals_all(params_now, X_obj_f_use)
            print(
                "Scaled PDE RMS on obj:",
                "phi", _rms(r_phi_obj),
                "Ts", _rms(r_Ts_obj),
                "cont", _rms(r_cont_obj),
                "ru", _rms(r_u_obj),
                "rv", _rms(r_v_obj),
                "rTf", _rms(r_Tf_obj),
            )

    print(f"[done] elapsed={time.time() - t0_clock:.2f}s")
    return theta

# ============================================================
# MAIN
# ============================================================
def main():
    theta = train_sqp(
        seed=0,
        hidden_dim=35,
        num_hidden=3,
        max_iters=5000,
        print_every=10,
        beta_shift=200.0,
        beta_power=0.6,
        alpha_cap=1e1,
        L_lip=40.0,
        Gamma_lip=40.0,
        jitter_every_iter=True,
        sx=5e-4,
        sy=5e-4,
        st=1e-4,
        sx_ic=5e-4,
        sy_ic=5e-4,
        sy_if=5e-4,
        st_if=1e-4,
        w_obj_phi=20.0,
        w_obj_Ts=10.0,
        w_obj_cont=20.0,
        w_obj_ru=100.0,
        w_obj_rv=100.0,
        w_obj_rTf=10.0,
    )

    np.save("theta_sqp_pinn_cleanv2.npy", np.array(theta))
    print("Saved theta -> theta_sqp_pinn_cleanv2.npy")

if __name__ == "__main__":
    main()

Loaded phi.txt: (65,) (16,) (65, 16) range= 0.5 6.2
Loaded tfluid truth: (4, 17, 12, 64)
M_CON = 2748 n_params = 2876
[it=10] obj=6.809e+01 feas_raw=6.810e-01 alpha=4.07e-03 beta=9.71e-01 station=1.61e+01 dn=1.61e+01
blocks (unscaled c):
  pde_phi   : rms=1.473e+00  max=4.238e+00
  pde_Ts    : rms=3.601e-01  max=9.575e-01
  pde_cont  : rms=5.704e-01  max=1.329e+00
  pde_ru    : rms=5.333e-01  max=1.665e+00
  pde_rv    : rms=4.798e-01  max=1.675e+00
  pde_rTf   : rms=3.373e-01  max=9.708e-01
  bc_phi    : rms=7.914e-01  max=1.806e+00
  bc_solid  : rms=2.025e-01  max=3.780e-01
  bc_fluid  : rms=9.939e-01  max=2.363e+00
  iface     : rms=7.635e-01  max=1.437e+00
  ic        : rms=8.801e-01  max=2.715e+00
Scaled PDE RMS on obj: phi 1.4274354407387606 Ts 0.35644422740156206 cont 0.5826962265709256 ru 0.5436238608327929 rv 0.49035854980356147 rTf 0.33071076360864704
[it=20] obj=5.817e+01 feas_raw=5.825e-01 alpha=1.24e-02 beta=9.44e-01 station=7.88e+00 dn=7.88e+00
blocks (unscaled c):
  pde_p